# RQ1 prototype (Silver): disease group and 180-day mortality

This prototype runs on the Silver table (`support2_cleaned.csv`) instead of Gold, because Silver still carries every column including `sex`. That lets the full `age + sex + scoma` model get built before sex is added to Gold. Once sex is in Gold, the final version only needs the load line switched to `support2_model_features.csv` and the manual leakage scrub dropped, since Gold is already clean.

In [ ]:
# Imports
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# Constants
RANDOM_STATE = 42
ALPHA = 0.05
REFERENCE_GROUP = "ARF/MOSF w/Sepsis"
np.random.seed(RANDOM_STATE)

# add repo root to path so utils imports (notebook runs from prototypes/)
sys.path.insert(0, str(Path.cwd().parent))
from utils.dataset import load_csv  # noqa: E402

## RQ1: does a patient's disease group shape their odds of dying within 180 days?

This is the project's first and primary research question, and it sets the baseline the rest of the analysis builds on. The study is organized around disease group, every later model holds it constant, and the disease-group signal is the strongest in the cohort. So pinning down this estimate is what the later physiology question and the predictive work lean on.

As an inference question, the model checks whether a patient's main illness is associated with the odds of death within 180 days, and by how much, after age, sex, and overall severity are held constant. Each odds ratio is read against the reference group, ARF/MOSF w/Sepsis (the largest, near-average group): above 1 means higher odds of death than the reference, below 1 means more survivable.

In [ ]:
# Load the Silver table (cleaned, still carries every column including sex).
silver = load_csv("support2_cleaned.csv")
print("rows, cols:", silver.shape)

# Base rate of the outcome (roughly half the cohort died within 180 days).
base_rate = silver["death_180d"].mean()
print(f"death_180d base rate: {base_rate:.3f} "
      f"({base_rate * 100:.1f}% died within 180 days)")

silver.head()

In [ ]:
# Columns we actually model: the disease group, the controls, and the target.
MODEL_COLS = ["death_180d", "age", "sex", "scoma", "dzgroup"]

# leakage and benchmark columns not in the model
# (picking MODEL_COLS already drops them; this just confirms none slipped through)
LEAKAGE_COLS = [
    "surv2m", "surv6m", "prg2m", "prg6m", "aps", "sps",
    "death", "hospdead", "d.time", "d_time", "slos",
    "dnr", "dnrday", "sfdm2",
]

model_df = silver[MODEL_COLS].dropna()
print("modeling frame shape:", model_df.shape)

leaked = [c for c in LEAKAGE_COLS if c in model_df.columns]
print("leakage/benchmark columns present:", leaked if leaked else "none (clean)")

model_df.head()

### The model

One multivariable logistic regression was fit:

    death_180d ~ age + sex + scoma + C(dzgroup, reference = "ARF/MOSF w/Sepsis")

- death_180d = died within 180 days (1 / 0), the outcome.
- age = per year, held constant as a confounder.
- sex = a demographic confounder, held constant.
- scoma = the day-3 coma score, the in-model severity control (aps and sps were
  pulled out as benchmarks, so scoma carries severity here).
- C(dzgroup, ...) = disease group as categories, each compared to the sepsis
  reference.

Logistic because the outcome is yes/no. Multivariable so the disease-group effect is measured after age, sex, and severity are accounted for, not before.

In [ ]:
# Fit the multivariable logistic regression with statsmodels. The reference
# group is fixed so every disease-group odds ratio is read against sepsis.
formula = (
    "death_180d ~ age + sex + scoma + "
    f'C(dzgroup, Treatment(reference="{REFERENCE_GROUP}"))'
)
model = smf.logit(formula, data=model_df).fit(disp=False)
print(model.summary())

In [ ]:
# Build the odds-ratio table from the fitted model.
params = model.params
conf = model.conf_int()
conf.columns = ["ci_low", "ci_high"]

or_table = pd.DataFrame({
    "OR": np.exp(params),
    "OR_ci_low": np.exp(conf["ci_low"]),
    "OR_ci_high": np.exp(conf["ci_high"]),
    "p_value": model.pvalues,
})

# keep just the disease-group terms for the FDR step and the forest plot
dz_mask = or_table.index.str.startswith("C(dzgroup")
dz_table = or_table[dz_mask].copy()

# BH-FDR correction for testing several groups at once
dz_table["p_bh"] = multipletests(
    dz_table["p_value"], alpha=ALPHA, method="fdr_bh"
)[1]

# pull the group name out of the long statsmodels label
dz_table.index = [name.split("T.")[-1].rstrip("]") for name in dz_table.index]
dz_table.index.name = "disease_group"

# sort by odds ratio, highest odds of death at the top
dz_table = dz_table.sort_values("OR", ascending=False)
dz_table[["OR", "OR_ci_low", "OR_ci_high", "p_value", "p_bh"]].round(4)

In [ ]:
# the controls (age, sex, scoma), shown next to the disease groups
control_terms = [t for t in or_table.index
                 if t in ("age", "scoma") or t.startswith("sex")]
or_table.loc[control_terms, ["OR", "OR_ci_low", "OR_ci_high", "p_value"]].round(4)

In [ ]:
# Labeled forest plot: one dot per disease group with its 95% CI bar.
fig, ax = plt.subplots(figsize=(8, 5))

groups = dz_table.index.tolist()
y = np.arange(len(groups))
or_vals = dz_table["OR"].to_numpy()
low = dz_table["OR_ci_low"].to_numpy()
high = dz_table["OR_ci_high"].to_numpy()

xerr = np.vstack([or_vals - low, high - or_vals])
ax.errorbar(
    or_vals, y, xerr=xerr, fmt="o", color="black",
    ecolor="gray", capsize=3, linestyle="none",
)

ax.axvline(1.0, linestyle="--", color="red", linewidth=1)
ax.set_xscale("log")
ax.set_yticks(y)
ax.set_yticklabels(groups)
ax.invert_yaxis()  # highest odds ratio on top
ax.set_xlabel("Odds ratio of death within 180 days (log scale)")
ax.set_ylabel("Disease group")
ax.set_title(
    "RQ1: disease-group odds of 180-day death\n"
    f"reference group = {REFERENCE_GROUP}; dashed line = no effect (OR = 1)"
)
fig.tight_layout()
plt.show()

### What the model says

The disease a patient came in with shaped the odds of dying within 180 days a
lot, even after holding age, sex, and severity constant. The highest-odds group
was MOSF w/Malig (multi-organ failure with cancer), at about 4.6 times the odds
of the sepsis reference group, followed by Lung Cancer at about 2.9 times.
Cirrhosis (about 1.6 times) and Colon Cancer (about 1.3 times) were also higher
than the reference. On the other end, CHF (heart failure) and COPD were the most
survivable, at about 0.54 and 0.61 times the reference odds.

Coma came out at about 1.2 times the reference, but its confidence interval
crossed 1 and it did not survive the BH-FDR correction (adjusted p = 0.12), so
this model cannot tell coma apart from the sepsis reference. That runs against an
earlier expectation that coma would come in high.

The two controls behaved as expected. Each extra year of age multiplied the odds
by about 1.02 (a 2.3 percent rise per year, p < 0.001), and each point of the
coma score raised the odds by about 1.03 (p < 0.001). Sex made only a small
difference: men had about 1.08 times the odds of women, and that gap was not
significant (p = 0.08) once disease, age, and severity were accounted for. That
is why sex is in the model as a control, not as one of the disease-group effects
the question is actually about: its own effect is small, but adjusting for it
keeps the disease-group odds ratios comparable.

Every disease-group difference except Coma stayed significant after the
Benjamini-Hochberg FDR correction, so these are not an artifact of testing
several groups at once.